# Landsat-8 Preprocessing Pipeline

End-to-end demonstration of `rs-toolkit` on a Landsat-8–like multispectral scene:

```
GeoTIFF → read_raster → cloud / nodata masking → normalization → NDVI / NDBI → 256×256 tiling → visualization
```

This notebook uses a **spectrally realistic synthetic scene** generated by `sample_data.make_landsat_scene()`.
The scene has real CRS (UTM 43N), 30 m pixel size, 6 Landsat-8 OLI bands (Blue, Green, Red, NIR, SWIR1, SWIR2),
scattered cloud pixels in the QA band, and nodata regions — so the full preprocessing pipeline is exercised.

> **To use real data:** replace the `scene_path` / `qa_path` below with paths to your actual Landsat-8 GeoTIFFs
> and adjust the band indices if needed.

## 1. Generate sample scene

In [ ]:
from rs_toolkit.sample_data import make_landsat_scene

scene_path, qa_path = make_landsat_scene(
    "sample_scene.tif",
    height=1024, width=1024,
    pixel_size=30.0,
    epsg=32643,
    cloud_frac=0.08,
    nodata_frac=0.02,
    seed=42,
)

## 2. Load and inspect

In [ ]:
from rs_toolkit.io import read_raster
from rs_toolkit.stats import band_stats, print_stats

arr, profile = read_raster(scene_path)
print(f"Shape: {arr.shape}  CRS: {profile['crs']}  Pixel size: {profile['transform'].a} m")
print()

band_names = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"]
print_stats(arr, band_names)

## 3. Visualize raw bands

In [ ]:
import matplotlib.pyplot as plt
from rs_toolkit.visualization import show, show_histogram

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
show(arr, bands=(2, 1, 0), title="True colour (R-G-B)", ax=axes[0])  # Red, Green, Blue
show(arr, bands=(3, 2, 1), title="False colour (NIR-R-G)", ax=axes[1])
show(arr[3:4], title="NIR band", ax=axes[2])
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
show_histogram(arr, band_labels=band_names, ax=ax)
plt.show()

## 4. Cloud and nodata masking

In [ ]:
import numpy as np
from rs_toolkit.preprocessing import nodata_mask, bitmask, apply_mask

# Read QA band
qa, _ = read_raster(qa_path)
qa_band = qa[0].astype(np.uint16)

# Build combined mask: nodata OR cloud (QA bit 4)
nd_mask = nodata_mask(arr)
cloud = bitmask(qa_band, bits=[4])
combined_mask = nd_mask | cloud

print(f"Nodata pixels:  {nd_mask.sum():,} ({100*nd_mask.mean():.1f}%)")
print(f"Cloud pixels:   {cloud.sum():,} ({100*cloud.mean():.1f}%)")
print(f"Combined mask:  {combined_mask.sum():,} ({100*combined_mask.mean():.1f}%)")

# Apply mask
arr_masked = apply_mask(arr, combined_mask)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(combined_mask, cmap="Reds")
axes[0].set_title("Cloud + nodata mask"); axes[0].axis("off")
show(arr_masked, bands=(2, 1, 0), title="Masked scene (RGB)", ax=axes[1])
fig.tight_layout()
plt.show()

## 5. Normalization

In [ ]:
from rs_toolkit.normalization import minmax, percentile_clip

arr_clip = percentile_clip(arr_masked, lo=2, hi=98)
arr_norm = minmax(arr_clip)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
show(arr_masked, bands=(2, 1, 0), title="Before normalization", ax=axes[0])
show(arr_norm, bands=(2, 1, 0), title="After percentile clip + min-max", ax=axes[1])
fig.tight_layout()
plt.show()

## 6. Spectral indices: NDVI and NDBI

In [ ]:
from rs_toolkit.indices import ndvi, ndbi
from rs_toolkit.visualization import show_index

# Landsat-8 band order: 0=Blue, 1=Green, 2=Red, 3=NIR, 4=SWIR1, 5=SWIR2
vi = ndvi(arr_masked[3], arr_masked[2])   # NIR, Red
bi = ndbi(arr_masked[4], arr_masked[3])   # SWIR1, NIR

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
show_index(vi, title="NDVI — vegetation", cmap="RdYlGn", ax=axes[0])
show_index(bi, title="NDBI — built-up", cmap="RdYlBu_r", ax=axes[1])
fig.tight_layout()
plt.show()

print(f"NDVI — mean: {np.nanmean(vi):.3f}  range: [{np.nanmin(vi):.3f}, {np.nanmax(vi):.3f}]")
print(f"NDBI — mean: {np.nanmean(bi):.3f}  range: [{np.nanmin(bi):.3f}, {np.nanmax(bi):.3f}]")

## 7. Tiling into 256 × 256 patches

In [ ]:
from rs_toolkit.tiling import extract_patches, spatial_split
from rs_toolkit.visualization import show_grid

patches, coords = extract_patches(arr_norm, size=256, overlap=32, max_nan_frac=0.1)
print(f"Extracted {len(patches)} patches of shape {patches.shape[1:]}")
print(f"Rejected patches with >10% NaN")

fig = show_grid(patches, n=9, bands=(2, 1, 0))
fig.suptitle("Sample patches (RGB)", fontsize=14, y=1.02)
plt.show()

## 8. Spatial train / validation / test split

**Why spatial splitting?**
Randomly splitting adjacent satellite patches produces overly optimistic evaluation
because spatially correlated samples appear in both training and test sets.
This toolkit uses **block-based spatial splitting**: the scene is divided into
coarse grid blocks, and entire blocks are assigned to train, val, or test
so that nearby patches never leak across splits.

In [ ]:
from rs_toolkit.visualization import show_split_map

train_idx, val_idx, test_idx = spatial_split(coords, size=256, block=512, seed=0)

print(f"Train: {len(train_idx)} patches")
print(f"Val:   {len(val_idx)} patches")
print(f"Test:  {len(test_idx)} patches")

fig, ax = plt.subplots(figsize=(8, 8))
show_split_map(coords, [train_idx, val_idx, test_idx], size=256,
               title="Spatial split (no leakage between blocks)", ax=ax)
plt.show()

## 9. Summary

This notebook demonstrated the full `rs-toolkit` preprocessing pipeline:

| Step | Function | Result |
|------|----------|--------|
| Load | `read_raster()` | 6-band (C,H,W) float32 array + georef profile |
| Mask | `nodata_mask()` + `bitmask()` + `apply_mask()` | NaN where cloud or nodata |
| Normalize | `percentile_clip()` + `minmax()` | [0,1] per band |
| Indices | `ndvi()`, `ndbi()` | Vegetation and built-up maps |
| Tile | `extract_patches(size=256, overlap=32)` | Clean 256×256 patches |
| Split | `spatial_split(block=512)` | Leakage-free train/val/test |

The output patches are ready for downstream ML training.

In [ ]:
# Clean up generated files
import os
for f in [scene_path, qa_path]:
    if os.path.exists(f): os.remove(f)
print("Cleaned up sample files.")